# Tgsp1 rescue: IL-1β ELISA analysis

LPS (200 ng/mL, 3 h) + nigericin (10 µM, 45 min) in Tgsp1-KO C4 iBMDM re-expressing Tgsp1-WT, Tgsp1-3A (PI4P-binding mutant) or empty vector. WT iBMDM as reference.
Plate: `raw/plate_2026-09-30_OD450.txt`. Experiments 1–3 (independent cultures), three ELISA wells each. Samples diluted 1:10.

In [ ]:
# ---- settings ----
INPUT_CSV = "data/elisa_rescue.csv"
STANDARDS_CSV = "data/standards.csv"
OUTPUT_PNG = "figure_rescue.png"

RECOMPUTE_FROM_OD = False   # True: pg/mL recalculated from OD450 with the standard curve
ONE_TAILED = True         # test for 3A vs Tgsp1-WT: False = two-sided

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

## 1. Standard curve (4-parameter logistic)

In [ ]:
def four_pl(c, a, d, c50, b):
    return a + (d - a) / (1 + (c50 / c) ** b)

def inv_four_pl(od, a, d, c50, b):
    return c50 / (((d - a) / (od - a)) - 1) ** (1 / b)

std = pd.read_csv(STANDARDS_CSV)
x = np.r_[std.conc_pg_ml, std.conc_pg_ml]
y = np.r_[std.od1, std.od2]
popt, _ = curve_fit(four_pl, x, y, p0=[0.05, 3, 800, 1], maxfev=20000)
r2 = 1 - np.sum((y - four_pl(x, *popt)) ** 2) / np.sum((y - y.mean()) ** 2)
print("4PL parameters (a, d, c50, b):", np.round(popt, 3), f"  R² = {r2:.4f}")

## 2. Load data and calculate concentrations

In [ ]:
df = pd.read_csv(INPUT_CSV)
if RECOMPUTE_FROM_OD:
    df["conc"] = inv_four_pl(df.od450, *popt) * df.dilution_factor
else:
    df["conc"] = df.conc_pg_ml
df.groupby(["condition", "experiment"]).conc.agg(["mean", "std", "count"]).round(1)

## 3. Replicate means and statistics
Means of the three wells per experiment are the unit of analysis. Welch's t-test.

In [ ]:
labels = {"WT_iBMDM": "WT\niBMDM", "KO_EV": "KO +\nempty vector",
          "KO_Tgsp1_WT": "KO +\nTgsp1-WT", "KO_Tgsp1_3A": "KO +\nTgsp1-3A"}
order = list(labels)

rep = df.groupby(["condition", "experiment"], as_index=False).conc.mean()
summary = rep.groupby("condition").conc.agg(mean="mean", sd="std", n="count").loc[order].round(1)
print(summary, "\n")

def welch(a, b, alternative="two-sided"):
    ga = rep.loc[rep.condition == a, "conc"]
    gb = rep.loc[rep.condition == b, "conc"]
    return stats.ttest_ind(ga, gb, equal_var=False, alternative=alternative).pvalue

tests = {
    ("KO_EV", "KO_Tgsp1_WT"):       welch("KO_Tgsp1_WT", "KO_EV"),
    ("KO_Tgsp1_WT", "KO_Tgsp1_3A"): welch("KO_Tgsp1_3A", "KO_Tgsp1_WT", "less" if ONE_TAILED else "two-sided"),
}
for (a, b), p in tests.items():
    print(f"{a:12s} vs {b:12s}  p = {p:.4f}")

m = summary["mean"]
print(f"\n3A relative to Tgsp1-WT: {m.KO_Tgsp1_3A / m.KO_Tgsp1_WT:.0%}")

## 4. Figure

In [ ]:
def pfmt(p):
    return "p < 0.001" if p < 0.001 else (f"p = {p:.3f}" if p < 0.05 else f"n.s. (p = {p:.2f})")

colors = {"WT_iBMDM": "#9a9993", "KO_EV": "#9a9993", "KO_Tgsp1_WT": "#2a78d6", "KO_Tgsp1_3A": "#eb6834"}
ink, ink2 = "#0b0b0b", "#52514e"

fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=200)
xs = np.arange(len(order))
for i, c in enumerate(order):
    vals = rep.loc[rep.condition == c, "conc"].to_numpy()
    ax.bar(i, vals.mean(), width=0.58, color=colors[c], alpha=0.85, zorder=2)
    ax.errorbar(i, vals.mean(), yerr=vals.std(ddof=1), color=ink, capsize=4, lw=1.2, zorder=3)
    ax.scatter(np.full(len(vals), i) + np.linspace(-0.12, 0.12, len(vals)), vals,
               s=22, color="white", edgecolor=ink, lw=1.0, zorder=4)

top = max(rep.groupby("condition").conc.apply(lambda v: v.mean() + (v.std(ddof=1) if len(v) > 1 else 0)))
step = top * 0.085
def bracket(i, j, level, text):
    y = top * 1.05 + level * step
    ax.plot([i, i, j, j], [y - step * 0.25, y, y, y - step * 0.25], color=ink2, lw=1)
    ax.text((i + j) / 2, y + step * 0.12, text, ha="center", va="bottom", fontsize=8, color=ink)

bracket(1, 2, 0, pfmt(tests[("KO_EV", "KO_Tgsp1_WT")]))
bracket(2, 3, 1, pfmt(tests[("KO_Tgsp1_WT", "KO_Tgsp1_3A")]))

ax.set_xticks(xs, [labels[c] for c in order], fontsize=9, color=ink)
ax.set_ylabel("IL-1β (pg/mL)", color=ink)
ax.set_ylim(0, top * 1.05 + 3.2 * step)
ax.yaxis.grid(True, color="#e3e2de", lw=0.8, zorder=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.tick_params(colors=ink2)
n = rep.groupby("condition").size()
ax.set_title("Rescue of IL-1β release in Tgsp1-KO iBMDM", loc="left", fontsize=11, color=ink)
fig.text(0.01, 0.01, f"LPS + nigericin. Bars: mean ± SD of experiment means; dots: experiments (n = {n.min()}–{n.max()} experiments). Welch's t-test.",
         fontsize=6.5, color=ink2)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUTPUT_PNG, facecolor="white")
plt.show()